# feature_kit — Usage Examples

This notebook walks through the core components of  using a synthetic dataset of 5 companies with 3 years of daily transaction history.

## 0. Generate Test Dataset

5 companies x 3 years (2021-01-01 to 2023-12-31) of daily observations.  
Metrics:  (daily revenue) and  (daily transaction count).

In [1]:
import polars as pl
import numpy as np
from datetime import date, timedelta

rng = np.random.default_rng(42)

companies = ["AAPL", "MSFT", "GOOGL", "AMZN", "META"]
start = date(2021, 1, 1)
n_days = 3 * 365
dates = [start + timedelta(days=i) for i in range(n_days)]

base_sales  = {"AAPL": 500, "MSFT": 400, "GOOGL": 350, "AMZN": 600, "META": 300}
base_count  = {"AAPL": 100, "MSFT":  80, "GOOGL":  70, "AMZN": 120, "META":  60}
growth_rate = {"AAPL": 0.10, "MSFT": 0.08, "GOOGL": 0.12, "AMZN": 0.15, "META": 0.05}

rows = []
for co in companies:
    for i, d in enumerate(dates):
        t = i / n_days
        trend = 1 + growth_rate[co] * t * 3
        sales = base_sales[co] * trend * (1 + 0.15 * rng.standard_normal())
        count = int(base_count[co] * trend * (1 + 0.10 * rng.standard_normal()))
        rows.append({"specid": co, "DATADATE": d, "SALES": round(float(sales), 2), "COUNT": float(max(count, 1))})

raw_df = pl.DataFrame(rows).sort(["specid", "DATADATE"])

print("Shape:", raw_df.shape)
print("Companies:", raw_df["specid"].unique().to_list())
print("Date range:", raw_df["DATADATE"].min(), "->", raw_df["DATADATE"].max())
raw_df.head(10)

Shape: (5475, 4)
Companies: ['AAPL', 'META', 'GOOGL', 'AMZN', 'MSFT']
Date range: 2021-01-01 -> 2023-12-31


specid,DATADATE,SALES,COUNT
str,date,f64,f64
"""AAPL""",2021-01-01,522.85,89.0
"""AAPL""",2021-01-02,556.44,109.0
"""AAPL""",2021-01-03,353.87,87.0
"""AAPL""",2021-01-04,510.01,96.0
"""AAPL""",2021-01-05,499.29,91.0
"""AAPL""",2021-01-06,566.73,107.0
"""AAPL""",2021-01-07,505.78,111.0
"""AAPL""",2021-01-08,536.09,91.0
"""AAPL""",2021-01-09,528.81,90.0


## 1. Params

In [2]:
from feature_kit import Param

params = Param(
    keys=["specid"],
    metrics=["SALES", "COUNT"],
)
print(params)

Param(keys=['specid'], metrics=['SALES', 'COUNT'], date_col='DATADATE')


## 2. Base Features

### 2a. Rolling Sum

In [3]:
from feature_kit import RollingSumBaseFeature

weekly_monthly_quarterly = [("weekly", 7), ("monthly", 28), ("quarterly", 91)]
rolling = RollingSumBaseFeature(params, windows=weekly_monthly_quarterly)
rolling_df = rolling.compute(raw_df)

print("Columns:", rolling_df.columns)
rolling_df.filter(pl.col("specid") == "AAPL").tail(5)

Columns: ['specid', 'DATADATE', 'SALES__ROLLING_SUM__WEEKLY', 'SALES__ROLLING_SUM__MONTHLY', 'SALES__ROLLING_SUM__QUARTERLY', 'COUNT__ROLLING_SUM__WEEKLY', 'COUNT__ROLLING_SUM__MONTHLY', 'COUNT__ROLLING_SUM__QUARTERLY']


specid,DATADATE,SALES__ROLLING_SUM__WEEKLY,SALES__ROLLING_SUM__MONTHLY,SALES__ROLLING_SUM__QUARTERLY,COUNT__ROLLING_SUM__WEEKLY,COUNT__ROLLING_SUM__MONTHLY,COUNT__ROLLING_SUM__QUARTERLY
str,date,f64,f64,f64,f64,f64,f64
"""AAPL""",2023-12-27,4703.33,18991.27,59491.85,871.0,3621.0,11626.0
"""AAPL""",2023-12-28,4594.67,18599.98,59366.06,866.0,3615.0,11624.0
"""AAPL""",2023-12-29,4508.82,18393.32,59167.46,881.0,3603.0,11624.0
"""AAPL""",2023-12-30,4527.07,18375.79,59302.34,891.0,3567.0,11606.0
"""AAPL""",2023-12-31,4691.44,18612.29,59519.84,898.0,3571.0,11630.0


### 2b. YoY Growth

In [4]:
from feature_kit import YOYGrowthBaseFeature

monthly_quarterly = [("monthly", 28), ("quarterly", 91)]
yoy = YOYGrowthBaseFeature(params, windows=monthly_quarterly)
yoy_df = yoy.compute(raw_df)

print("Columns:", yoy_df.columns)
yoy_df.filter(pl.col("specid") == "AAPL").drop_nulls().head(5)

Columns: ['specid', 'DATADATE', 'SALES__YOY_GROWTH__MONTHLY', 'SALES__YOY_GROWTH__QUARTERLY', 'COUNT__YOY_GROWTH__MONTHLY', 'COUNT__YOY_GROWTH__QUARTERLY']


specid,DATADATE,SALES__YOY_GROWTH__MONTHLY,SALES__YOY_GROWTH__QUARTERLY,COUNT__YOY_GROWTH__MONTHLY,COUNT__YOY_GROWTH__QUARTERLY
str,date,f64,f64,f64,f64
"""AAPL""",2021-12-31,27.625954,92.079277,32.876404,107.898876
"""AAPL""",2022-01-01,12.841859,44.084185,14.212121,47.929293
"""AAPL""",2022-01-02,9.322127,32.970813,9.547368,32.97193
"""AAPL""",2022-01-03,6.610384,24.060659,6.900262,24.364829
"""AAPL""",2022-01-04,5.055104,18.972004,5.381356,19.504237


### 2c. Period-over-Period Growth

In [5]:
from feature_kit import POPGrowthBaseFeature

pop = POPGrowthBaseFeature(params, windows=monthly_quarterly)
pop_df = pop.compute(raw_df)

print("Columns:", pop_df.columns)
pop_df.filter(pl.col("specid") == "AAPL").drop_nulls().head(5)

Columns: ['specid', 'DATADATE', 'SALES__POP_GROWTH__MONTHLY', 'COUNT__POP_GROWTH__MONTHLY', 'SALES__POP_GROWTH__QUARTERLY', 'COUNT__POP_GROWTH__QUARTERLY']


specid,DATADATE,SALES__POP_GROWTH__MONTHLY,COUNT__POP_GROWTH__MONTHLY,SALES__POP_GROWTH__QUARTERLY,COUNT__POP_GROWTH__QUARTERLY
str,date,f64,f64,f64,f64
"""AAPL""",2021-04-02,0.015366,0.01581,86.841809,101.41573
"""AAPL""",2021-04-03,0.019688,0.030336,41.543913,45.080808
"""AAPL""",2021-04-04,0.025775,0.038559,31.159829,31.021053
"""AAPL""",2021-04-05,0.008942,0.046977,22.666375,23.013123
"""AAPL""",2021-04-06,0.038134,0.052555,17.888395,18.402542


## 3. Benchmarks

### 3a. Zero Benchmark

In [6]:
from feature_kit import ZeroBenchmark

zero_bench = ZeroBenchmark(params)
zero_df = zero_bench.compute(yoy_df, raw_df)
zero_df.head(5)

specid,DATADATE,ZERO_BENCHMARK
str,date,f64
"""AAPL""",2021-01-01,0.0
"""AAPL""",2021-01-02,0.0
"""AAPL""",2021-01-03,0.0
"""AAPL""",2021-01-04,0.0
"""AAPL""",2021-01-05,0.0


### 3b. Lagged YoY Growth Benchmark

The dev-branch API uses  and  to generate multiples of each period.

In [7]:
from feature_kit import LaggedYOYGrowthBenchmark

month_quarter = [("month", 28), ("quarter", 91)]
lagged_bench = LaggedYOYGrowthBenchmark(params, periods=month_quarter, num_lags=2)
lagged_df = lagged_bench.compute(yoy_df, raw_df)

print("Columns:", lagged_df.columns)
# LAG1MONTH = 28d ago, LAG2MONTH = 56d ago, LAG1QUARTER = 91d ago, LAG2QUARTER = 182d ago
lagged_df.filter(pl.col("specid") == "AAPL").drop_nulls().head(5)

Columns: ['specid', 'DATADATE', 'SALES__YOY_GROWTH__MONTHLY__LAG1MONTH', 'SALES__YOY_GROWTH__QUARTERLY__LAG1MONTH', 'COUNT__YOY_GROWTH__MONTHLY__LAG1MONTH', 'COUNT__YOY_GROWTH__QUARTERLY__LAG1MONTH', 'SALES__YOY_GROWTH__MONTHLY__LAG2MONTH', 'SALES__YOY_GROWTH__QUARTERLY__LAG2MONTH', 'COUNT__YOY_GROWTH__MONTHLY__LAG2MONTH', 'COUNT__YOY_GROWTH__QUARTERLY__LAG2MONTH', 'SALES__YOY_GROWTH__MONTHLY__LAG1QUARTER', 'SALES__YOY_GROWTH__QUARTERLY__LAG1QUARTER', 'COUNT__YOY_GROWTH__MONTHLY__LAG1QUARTER', 'COUNT__YOY_GROWTH__QUARTERLY__LAG1QUARTER', 'SALES__YOY_GROWTH__MONTHLY__LAG2QUARTER', 'SALES__YOY_GROWTH__QUARTERLY__LAG2QUARTER', 'COUNT__YOY_GROWTH__MONTHLY__LAG2QUARTER', 'COUNT__YOY_GROWTH__QUARTERLY__LAG2QUARTER']


specid,DATADATE,SALES__YOY_GROWTH__MONTHLY__LAG1MONTH,SALES__YOY_GROWTH__QUARTERLY__LAG1MONTH,COUNT__YOY_GROWTH__MONTHLY__LAG1MONTH,COUNT__YOY_GROWTH__QUARTERLY__LAG1MONTH,SALES__YOY_GROWTH__MONTHLY__LAG2MONTH,SALES__YOY_GROWTH__QUARTERLY__LAG2MONTH,COUNT__YOY_GROWTH__MONTHLY__LAG2MONTH,COUNT__YOY_GROWTH__QUARTERLY__LAG2MONTH,SALES__YOY_GROWTH__MONTHLY__LAG1QUARTER,SALES__YOY_GROWTH__QUARTERLY__LAG1QUARTER,COUNT__YOY_GROWTH__MONTHLY__LAG1QUARTER,COUNT__YOY_GROWTH__QUARTERLY__LAG1QUARTER,SALES__YOY_GROWTH__MONTHLY__LAG2QUARTER,SALES__YOY_GROWTH__QUARTERLY__LAG2QUARTER,COUNT__YOY_GROWTH__MONTHLY__LAG2QUARTER,COUNT__YOY_GROWTH__QUARTERLY__LAG2QUARTER
str,date,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64
"""AAPL""",2022-07-01,0.057894,0.077596,0.13324,0.102404,0.11778,0.103112,0.098631,0.093441,0.030218,0.096673,0.085249,0.103236,27.625954,92.079277,32.876404,107.898876
"""AAPL""",2022-07-02,0.053344,0.075216,0.134427,0.101044,0.121607,0.103823,0.096187,0.095347,0.025622,0.097421,0.071504,0.1016,12.841859,44.084185,14.212121,47.929293
"""AAPL""",2022-07-03,0.038652,0.073119,0.132614,0.099205,0.12459,0.103314,0.105597,0.096844,0.039608,0.094398,0.072855,0.103331,9.322127,32.970813,9.547368,32.97193
"""AAPL""",2022-07-04,0.045314,0.075739,0.12721,0.101116,0.117467,0.097846,0.113435,0.097744,0.0652,0.099243,0.073391,0.101323,6.610384,24.060659,6.900262,24.364829
"""AAPL""",2022-07-05,0.041074,0.071775,0.126995,0.099549,0.097866,0.09907,0.106066,0.098584,0.055425,0.097129,0.06276,0.097292,5.055104,18.972004,5.381356,19.504237


### 3c. Cohort Rolling Sum Benchmark

In [8]:
from feature_kit import CohortRollingSumBenchmark

cohort_raw = raw_df.with_columns(pl.lit("ALL").alias("cohort"))

rolling_for_cohort = RollingSumBaseFeature(params, windows=[("monthly", 28)]).compute(cohort_raw)
rolling_for_cohort = rolling_for_cohort.join(
    cohort_raw.select(["specid", "DATADATE", "cohort"]), on=["specid", "DATADATE"]
)

cohort_bench = CohortRollingSumBenchmark(params, cohort_cols=["cohort"], agg="mean")
cohort_df = cohort_bench.compute(rolling_for_cohort, cohort_raw)

print("Columns:", cohort_df.columns)
cohort_df.filter(pl.col("specid") == "AAPL").head(5)

Columns: ['specid', 'DATADATE', 'SALES__ROLLING_SUM__MONTHLY__COHORT_BENCHMARK', 'COUNT__ROLLING_SUM__MONTHLY__COHORT_BENCHMARK']


specid,DATADATE,SALES__ROLLING_SUM__MONTHLY__COHORT_BENCHMARK,COUNT__ROLLING_SUM__MONTHLY__COHORT_BENCHMARK
str,date,f64,f64
"""AAPL""",2021-01-01,463.772,85.6
"""AAPL""",2021-01-02,934.762,168.6
"""AAPL""",2021-01-03,1349.804,248.4
"""AAPL""",2021-01-04,1791.428,330.2
"""AAPL""",2021-01-05,2168.516,416.4


## 4. Comparisons

### 4a. Difference & Sign

In [9]:
from feature_kit import DifferenceComparison, SignComparison

diff = DifferenceComparison().compute(yoy_df, zero_df)
sign = SignComparison().compute(yoy_df, zero_df)

print("Difference columns:", diff.columns)
diff.filter(pl.col("specid") == "AAPL").drop_nulls().head(5)

Difference columns: ['specid', 'DATADATE', 'SALES__YOY_GROWTH__MONTHLY__VS__ZERO_BENCHMARK__DIFFERENCE', 'SALES__YOY_GROWTH__QUARTERLY__VS__ZERO_BENCHMARK__DIFFERENCE', 'COUNT__YOY_GROWTH__MONTHLY__VS__ZERO_BENCHMARK__DIFFERENCE', 'COUNT__YOY_GROWTH__QUARTERLY__VS__ZERO_BENCHMARK__DIFFERENCE']


specid,DATADATE,SALES__YOY_GROWTH__MONTHLY__VS__ZERO_BENCHMARK__DIFFERENCE,SALES__YOY_GROWTH__QUARTERLY__VS__ZERO_BENCHMARK__DIFFERENCE,COUNT__YOY_GROWTH__MONTHLY__VS__ZERO_BENCHMARK__DIFFERENCE,COUNT__YOY_GROWTH__QUARTERLY__VS__ZERO_BENCHMARK__DIFFERENCE
str,date,f64,f64,f64,f64
"""AAPL""",2021-12-31,27.625954,92.079277,32.876404,107.898876
"""AAPL""",2022-01-01,12.841859,44.084185,14.212121,47.929293
"""AAPL""",2022-01-02,9.322127,32.970813,9.547368,32.97193
"""AAPL""",2022-01-03,6.610384,24.060659,6.900262,24.364829
"""AAPL""",2022-01-04,5.055104,18.972004,5.381356,19.504237


In [10]:
print("Sign columns:", sign.columns)
sign.filter(pl.col("specid") == "AAPL").drop_nulls().head(5)

Sign columns: ['specid', 'DATADATE', 'SALES__YOY_GROWTH__MONTHLY__VS__ZERO_BENCHMARK__SIGN', 'SALES__YOY_GROWTH__QUARTERLY__VS__ZERO_BENCHMARK__SIGN', 'COUNT__YOY_GROWTH__MONTHLY__VS__ZERO_BENCHMARK__SIGN', 'COUNT__YOY_GROWTH__QUARTERLY__VS__ZERO_BENCHMARK__SIGN']


specid,DATADATE,SALES__YOY_GROWTH__MONTHLY__VS__ZERO_BENCHMARK__SIGN,SALES__YOY_GROWTH__QUARTERLY__VS__ZERO_BENCHMARK__SIGN,COUNT__YOY_GROWTH__MONTHLY__VS__ZERO_BENCHMARK__SIGN,COUNT__YOY_GROWTH__QUARTERLY__VS__ZERO_BENCHMARK__SIGN
str,date,f64,f64,f64,f64
"""AAPL""",2021-12-31,1.0,1.0,1.0,1.0
"""AAPL""",2022-01-01,1.0,1.0,1.0,1.0
"""AAPL""",2022-01-02,1.0,1.0,1.0,1.0
"""AAPL""",2022-01-03,1.0,1.0,1.0,1.0
"""AAPL""",2022-01-04,1.0,1.0,1.0,1.0


### 4b. Rolling Z-Score

In [11]:
from feature_kit import ZScoreComparison

zscore = ZScoreComparison(params, lookback_window=91).compute(yoy_df, zero_df)

print("ZScore columns:", zscore.columns)
zscore.filter(pl.col("specid") == "AAPL").drop_nulls().head(5)

ZScore columns: ['specid', 'DATADATE', 'SALES__YOY_GROWTH__MONTHLY__VS__ZERO_BENCHMARK__ZSCORE', 'SALES__YOY_GROWTH__QUARTERLY__VS__ZERO_BENCHMARK__ZSCORE', 'COUNT__YOY_GROWTH__MONTHLY__VS__ZERO_BENCHMARK__ZSCORE', 'COUNT__YOY_GROWTH__QUARTERLY__VS__ZERO_BENCHMARK__ZSCORE']


specid,DATADATE,SALES__YOY_GROWTH__MONTHLY__VS__ZERO_BENCHMARK__ZSCORE,SALES__YOY_GROWTH__QUARTERLY__VS__ZERO_BENCHMARK__ZSCORE,COUNT__YOY_GROWTH__MONTHLY__VS__ZERO_BENCHMARK__ZSCORE,COUNT__YOY_GROWTH__QUARTERLY__VS__ZERO_BENCHMARK__ZSCORE
str,date,f64,f64,f64,f64
"""AAPL""",2022-03-31,-0.293939,-0.374284,-0.266429,-0.352397
"""AAPL""",2022-04-01,-0.364148,-0.488089,-0.340182,-0.482847
"""AAPL""",2022-04-02,-0.3967,-0.542089,-0.376993,-0.548553
"""AAPL""",2022-04-03,-0.41828,-0.600379,-0.398317,-0.6017
"""AAPL""",2022-04-04,-0.413884,-0.644827,-0.413704,-0.644186


### 4c. Cross-Sectional Rank

In [12]:
from feature_kit import RankComparison

rank = RankComparison(params, lookback_window=91).compute(yoy_df, zero_df)

rank_col = [c for c in rank.columns if "__RANK" in c][0]
print(f"Column: {rank_col}")
vmin = rank[rank_col].drop_nulls().min()
vmax = rank[rank_col].drop_nulls().max()
print(f"Range: [{vmin:.3f}, {vmax:.3f}]")


Column: SALES__YOY_GROWTH__MONTHLY__VS__ZERO_BENCHMARK__RANK
Range: [-0.978, 1.000]


### 4d. OLS Residual

In [13]:
from feature_kit import ResidualComparison

residual = ResidualComparison(params, lookback_window=91).compute(yoy_df, lagged_df)

print("Residual columns:", residual.columns)
residual.filter(pl.col("specid") == "AAPL").drop_nulls().head(5)

Residual columns: ['specid', 'DATADATE', 'SALES__YOY_GROWTH__MONTHLY__VS__SALES__YOY_GROWTH__MONTHLY__LAG1MONTH__RESIDUAL', 'SALES__YOY_GROWTH__MONTHLY__VS__SALES__YOY_GROWTH__QUARTERLY__LAG1MONTH__RESIDUAL', 'SALES__YOY_GROWTH__MONTHLY__VS__SALES__YOY_GROWTH__MONTHLY__LAG2MONTH__RESIDUAL', 'SALES__YOY_GROWTH__MONTHLY__VS__SALES__YOY_GROWTH__QUARTERLY__LAG2MONTH__RESIDUAL', 'SALES__YOY_GROWTH__MONTHLY__VS__SALES__YOY_GROWTH__MONTHLY__LAG1QUARTER__RESIDUAL', 'SALES__YOY_GROWTH__MONTHLY__VS__SALES__YOY_GROWTH__QUARTERLY__LAG1QUARTER__RESIDUAL', 'SALES__YOY_GROWTH__MONTHLY__VS__SALES__YOY_GROWTH__MONTHLY__LAG2QUARTER__RESIDUAL', 'SALES__YOY_GROWTH__MONTHLY__VS__SALES__YOY_GROWTH__QUARTERLY__LAG2QUARTER__RESIDUAL', 'SALES__YOY_GROWTH__QUARTERLY__VS__SALES__YOY_GROWTH__MONTHLY__LAG1MONTH__RESIDUAL', 'SALES__YOY_GROWTH__QUARTERLY__VS__SALES__YOY_GROWTH__QUARTERLY__LAG1MONTH__RESIDUAL', 'SALES__YOY_GROWTH__QUARTERLY__VS__SALES__YOY_GROWTH__MONTHLY__LAG2MONTH__RESIDUAL', 'SALES__YOY_GROWTH__QU

specid,DATADATE,SALES__YOY_GROWTH__MONTHLY__VS__SALES__YOY_GROWTH__MONTHLY__LAG1MONTH__RESIDUAL,SALES__YOY_GROWTH__MONTHLY__VS__SALES__YOY_GROWTH__QUARTERLY__LAG1MONTH__RESIDUAL,SALES__YOY_GROWTH__MONTHLY__VS__SALES__YOY_GROWTH__MONTHLY__LAG2MONTH__RESIDUAL,SALES__YOY_GROWTH__MONTHLY__VS__SALES__YOY_GROWTH__QUARTERLY__LAG2MONTH__RESIDUAL,SALES__YOY_GROWTH__MONTHLY__VS__SALES__YOY_GROWTH__MONTHLY__LAG1QUARTER__RESIDUAL,SALES__YOY_GROWTH__MONTHLY__VS__SALES__YOY_GROWTH__QUARTERLY__LAG1QUARTER__RESIDUAL,SALES__YOY_GROWTH__MONTHLY__VS__SALES__YOY_GROWTH__MONTHLY__LAG2QUARTER__RESIDUAL,SALES__YOY_GROWTH__MONTHLY__VS__SALES__YOY_GROWTH__QUARTERLY__LAG2QUARTER__RESIDUAL,SALES__YOY_GROWTH__QUARTERLY__VS__SALES__YOY_GROWTH__MONTHLY__LAG1MONTH__RESIDUAL,SALES__YOY_GROWTH__QUARTERLY__VS__SALES__YOY_GROWTH__QUARTERLY__LAG1MONTH__RESIDUAL,SALES__YOY_GROWTH__QUARTERLY__VS__SALES__YOY_GROWTH__MONTHLY__LAG2MONTH__RESIDUAL,SALES__YOY_GROWTH__QUARTERLY__VS__SALES__YOY_GROWTH__QUARTERLY__LAG2MONTH__RESIDUAL,SALES__YOY_GROWTH__QUARTERLY__VS__SALES__YOY_GROWTH__MONTHLY__LAG1QUARTER__RESIDUAL,SALES__YOY_GROWTH__QUARTERLY__VS__SALES__YOY_GROWTH__QUARTERLY__LAG1QUARTER__RESIDUAL,SALES__YOY_GROWTH__QUARTERLY__VS__SALES__YOY_GROWTH__MONTHLY__LAG2QUARTER__RESIDUAL,SALES__YOY_GROWTH__QUARTERLY__VS__SALES__YOY_GROWTH__QUARTERLY__LAG2QUARTER__RESIDUAL,COUNT__YOY_GROWTH__MONTHLY__VS__COUNT__YOY_GROWTH__MONTHLY__LAG1MONTH__RESIDUAL,COUNT__YOY_GROWTH__MONTHLY__VS__COUNT__YOY_GROWTH__QUARTERLY__LAG1MONTH__RESIDUAL,COUNT__YOY_GROWTH__MONTHLY__VS__COUNT__YOY_GROWTH__MONTHLY__LAG2MONTH__RESIDUAL,COUNT__YOY_GROWTH__MONTHLY__VS__COUNT__YOY_GROWTH__QUARTERLY__LAG2MONTH__RESIDUAL,COUNT__YOY_GROWTH__MONTHLY__VS__COUNT__YOY_GROWTH__MONTHLY__LAG1QUARTER__RESIDUAL,COUNT__YOY_GROWTH__MONTHLY__VS__COUNT__YOY_GROWTH__QUARTERLY__LAG1QUARTER__RESIDUAL,COUNT__YOY_GROWTH__MONTHLY__VS__COUNT__YOY_GROWTH__MONTHLY__LAG2QUARTER__RESIDUAL,COUNT__YOY_GROWTH__MONTHLY__VS__COUNT__YOY_GROWTH__QUARTERLY__LAG2QUARTER__RESIDUAL,COUNT__YOY_GROWTH__QUARTERLY__VS__COUNT__YOY_GROWTH__MONTHLY__LAG1MONTH__RESIDUAL,COUNT__YOY_GROWTH__QUARTERLY__VS__COUNT__YOY_GROWTH__QUARTERLY__LAG1MONTH__RESIDUAL,COUNT__YOY_GROWTH__QUARTERLY__VS__COUNT__YOY_GROWTH__MONTHLY__LAG2MONTH__RESIDUAL,COUNT__YOY_GROWTH__QUARTERLY__VS__COUNT__YOY_GROWTH__QUARTERLY__LAG2MONTH__RESIDUAL,COUNT__YOY_GROWTH__QUARTERLY__VS__COUNT__YOY_GROWTH__MONTHLY__LAG1QUARTER__RESIDUAL,COUNT__YOY_GROWTH__QUARTERLY__VS__COUNT__YOY_GROWTH__QUARTERLY__LAG1QUARTER__RESIDUAL,COUNT__YOY_GROWTH__QUARTERLY__VS__COUNT__YOY_GROWTH__MONTHLY__LAG2QUARTER__RESIDUAL,COUNT__YOY_GROWTH__QUARTERLY__VS__COUNT__YOY_GROWTH__QUARTERLY__LAG2QUARTER__RESIDUAL
str,date,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64
"""AAPL""",2022-09-29,0.062107,0.079141,0.072133,0.070242,0.080254,0.080512,0.067194,0.066717,0.039224,0.037775,0.027639,0.038833,0.030475,0.028255,0.038743,0.038319,0.041618,0.008071,0.018059,0.03102,0.020811,0.033833,0.018901,0.019963,0.005879,-0.001686,-0.011869,-0.009768,-0.016337,-0.000003,-0.018836,-0.018273
"""AAPL""",2022-09-30,0.066873,0.07914,0.077398,0.077989,0.086137,0.086054,0.074424,0.074216,0.035353,0.035396,0.021238,0.0352,0.027085,0.026034,0.03462,0.033713,0.036138,0.01048,0.018774,0.029591,0.02328,0.032224,0.020565,0.022677,0.005025,-0.001121,-0.007688,-0.007475,-0.011548,0.001548,-0.015137,-0.013813
"""AAPL""",2022-10-01,0.056656,0.069058,0.064247,0.06684,0.073406,0.074217,0.063738,0.064092,0.032192,0.031959,0.019716,0.031748,0.024154,0.022843,0.031238,0.029966,0.033601,0.012459,0.019776,0.030111,0.027661,0.038087,0.021698,0.024237,0.002405,-0.001579,-0.009156,-0.008097,-0.009233,0.008505,-0.015782,-0.013954
"""AAPL""",2022-10-02,0.044532,0.069903,0.056785,0.060109,0.06284,0.067203,0.058256,0.059125,0.028831,0.026563,0.016342,0.030242,0.026641,0.020623,0.027931,0.02614,0.030817,0.01094,0.017404,0.027628,0.022722,0.030406,0.019428,0.022352,0.004047,0.0000

## 5. Transformations

### 5a. Winsorize

In [14]:
from feature_kit import WinsorizeTransformation

winsorized = WinsorizeTransformation(lower_q=0.01, upper_q=0.99).compute(diff)

feat_col = [c for c in diff.columns if "__VS__" in c][0]
b_min = diff[feat_col].drop_nulls().min()
b_max = diff[feat_col].drop_nulls().max()
a_min = winsorized[feat_col].drop_nulls().min()
a_max = winsorized[feat_col].drop_nulls().max()
print(f"Column: {feat_col}")
print(f"Before: min={b_min:.4f}, max={b_max:.4f}")
print(f"After:  min={a_min:.4f}, max={a_max:.4f}")

Column: SALES__YOY_GROWTH__MONTHLY__VS__ZERO_BENCHMARK__DIFFERENCE
Before: min=-0.0857, max=34.9344
After:  min=-0.0106, max=2.7568


### 5b. Ramp Up

Down-weight features for new entities in their first year.

In [15]:
from feature_kit import RampUpTransformation

age_lookup = raw_df.group_by("specid").agg(pl.col("DATADATE").min().alias("first_date"))
diff_with_age = (
    diff
    .join(age_lookup, on="specid")
    .with_columns(
        ((pl.col("DATADATE") - pl.col("first_date")).dt.total_days()).alias("age_days")
    )
    .drop("first_date")
)

ramped = RampUpTransformation(age_col="age_days", min_age=0, max_age=365).compute(diff_with_age)

feat_col = [c for c in diff.columns if "__VS__" in c][0]
ramped.select(["specid", "DATADATE", "age_days", feat_col]).filter(
    pl.col("specid") == "AAPL"
).drop_nulls().head(8)

specid,DATADATE,age_days,SALES__YOY_GROWTH__MONTHLY__VS__ZERO_BENCHMARK__DIFFERENCE
str,date,i64,f64
"""AAPL""",2021-12-31,364,27.550266
"""AAPL""",2022-01-01,365,12.841859
"""AAPL""",2022-01-02,366,9.322127
"""AAPL""",2022-01-03,367,6.610384
"""AAPL""",2022-01-04,368,5.055104
"""AAPL""",2022-01-05,369,3.837438
"""AAPL""",2022-01-06,370,3.176158
"""AAPL""",2022-01-07,371,2.587626


## 6. Full Pipeline with FeatureBuilder

In [16]:
from feature_kit import (
    FeatureBuilder, Param,
    YOYGrowthBaseFeature,
    ZeroBenchmark, LaggedYOYGrowthBenchmark,
    DifferenceComparison, ZScoreComparison,
    WinsorizeTransformation,
)

params = Param(keys=["specid"], metrics=["SALES", "COUNT"])

builder = FeatureBuilder(
    params=params,
    base_feature=YOYGrowthBaseFeature(params, windows=[("monthly", 28), ("quarterly", 91)]),
    benchmarks=[
        ZeroBenchmark(params),
        LaggedYOYGrowthBenchmark(params, periods=[("month", 28), ("quarter", 91)], num_lags=2),
    ],
    comparisons=[
        DifferenceComparison(),
        ZScoreComparison(params, lookback_window=91),
    ],
    transformations=[
        WinsorizeTransformation(lower_q=0.01, upper_q=0.99),
    ],
)

result = builder.compute(raw_df)

print("Output shape:", result.shape)
print("\nOutput columns ({len(result.columns)} total):")
for col in result.columns:
    print(" ", col)

Output shape: (5475, 106)

Output columns ({len(result.columns)} total):
  specid
  DATADATE
  SALES__YOY_GROWTH__MONTHLY__VS__ZERO_BENCHMARK__DIFFERENCE
  SALES__YOY_GROWTH__QUARTERLY__VS__ZERO_BENCHMARK__DIFFERENCE
  COUNT__YOY_GROWTH__MONTHLY__VS__ZERO_BENCHMARK__DIFFERENCE
  COUNT__YOY_GROWTH__QUARTERLY__VS__ZERO_BENCHMARK__DIFFERENCE
  SALES__YOY_GROWTH__MONTHLY__VS__ZERO_BENCHMARK__ZSCORE
  SALES__YOY_GROWTH__QUARTERLY__VS__ZERO_BENCHMARK__ZSCORE
  COUNT__YOY_GROWTH__MONTHLY__VS__ZERO_BENCHMARK__ZSCORE
  COUNT__YOY_GROWTH__QUARTERLY__VS__ZERO_BENCHMARK__ZSCORE
  SALES__YOY_GROWTH__MONTHLY__VS__SALES__YOY_GROWTH__MONTHLY__LAG1MONTH__DIFFERENCE
  SALES__YOY_GROWTH__MONTHLY__VS__SALES__YOY_GROWTH__QUARTERLY__LAG1MONTH__DIFFERENCE
  SALES__YOY_GROWTH__MONTHLY__VS__COUNT__YOY_GROWTH__MONTHLY__LAG1MONTH__DIFFERENCE
  SALES__YOY_GROWTH__MONTHLY__VS__COUNT__YOY_GROWTH__QUARTERLY__LAG1MONTH__DIFFERENCE
  SALES__YOY_GROWTH__MONTHLY__VS__SALES__YOY_GROWTH__MONTHLY__LAG2MONTH__DIFFERENCE
  S

In [17]:
result.filter(pl.col("specid") == "AAPL").drop_nulls().head(5)

specid,DATADATE,SALES__YOY_GROWTH__MONTHLY__VS__ZERO_BENCHMARK__DIFFERENCE,SALES__YOY_GROWTH__QUARTERLY__VS__ZERO_BENCHMARK__DIFFERENCE,COUNT__YOY_GROWTH__MONTHLY__VS__ZERO_BENCHMARK__DIFFERENCE,COUNT__YOY_GROWTH__QUARTERLY__VS__ZERO_BENCHMARK__DIFFERENCE,SALES__YOY_GROWTH__MONTHLY__VS__ZERO_BENCHMARK__ZSCORE,SALES__YOY_GROWTH__QUARTERLY__VS__ZERO_BENCHMARK__ZSCORE,COUNT__YOY_GROWTH__MONTHLY__VS__ZERO_BENCHMARK__ZSCORE,COUNT__YOY_GROWTH__QUARTERLY__VS__ZERO_BENCHMARK__ZSCORE,SALES__YOY_GROWTH__MONTHLY__VS__SALES__YOY_GROWTH__MONTHLY__LAG1MONTH__DIFFERENCE,SALES__YOY_GROWTH__MONTHLY__VS__SALES__YOY_GROWTH__QUARTERLY__LAG1MONTH__DIFFERENCE,SALES__YOY_GROWTH__MONTHLY__VS__COUNT__YOY_GROWTH__MONTHLY__LAG1MONTH__DIFFERENCE,SALES__YOY_GROWTH__MONTHLY__VS__COUNT__YOY_GROWTH__QUARTERLY__LAG1MONTH__DIFFERENCE,SALES__YOY_GROWTH__MONTHLY__VS__SALES__YOY_GROWTH__MONTHLY__LAG2MONTH__DIFFERENCE,SALES__YOY_GROWTH__MONTHLY__VS__SALES__YOY_GROWTH__QUARTERLY__LAG2MONTH__DIFFERENCE,SALES__YOY_GROWTH__MONTHLY__VS__COUNT__YOY_GROWTH__MONTHLY__LAG2MONTH__DIFFERENCE,SALES__YOY_GROWTH__MONTHLY__VS__COUNT__YOY_GROWTH__QUARTERLY__LAG2MONTH__DIFFERENCE,SALES__YOY_GROWTH__MONTHLY__VS__SALES__YOY_GROWTH__MONTHLY__LAG1QUARTER__DIFFERENCE,SALES__YOY_GROWTH__MONTHLY__VS__SALES__YOY_GROWTH__QUARTERLY__LAG1QUARTER__DIFFERENCE,SALES__YOY_GROWTH__MONTHLY__VS__COUNT__YOY_GROWTH__MONTHLY__LAG1QUARTER__DIFFERENCE,SALES__YOY_GROWTH__MONTHLY__VS__COUNT__YOY_GROWTH__QUARTERLY__LAG1QUARTER__DIFFERENCE,SALES__YOY_GROWTH__MONTHLY__VS__SALES__YOY_GROWTH__MONTHLY__LAG2QUARTER__DIFFERENCE,SALES__YOY_GROWTH__MONTHLY__VS__SALES__YOY_GROWTH__QUARTERLY__LAG2QUARTER__DIFFERENCE,SALES__YOY_GROWTH__MONTHLY__VS__COUNT__YOY_GROWTH__MONTHLY__LAG2QUARTER__DIFFERENCE,SALES__YOY_GROWTH__MONTHLY__VS__COUNT__YOY_GROWTH__QUARTERLY__LAG2QUARTER__DIFFERENCE,SALES__YOY_GROWTH__QUARTERLY__VS__SALES__YOY_GROWTH__MONTHLY__LAG1MONTH__DIFFERENCE,SALES__YOY_GROWTH__QUARTERLY__VS__SALES__YOY_GROWTH__QUARTERLY__LAG1MONTH__DIFFERENCE,SALES__YOY_GROWTH__QUARTERLY__VS__COUNT__YOY_GROWTH__MONTHLY__LAG1MONTH__DIFFERENCE,SALES__YOY_GROWTH__QUARTERLY__VS__COUNT__YOY_GROWTH__QUARTERLY__LAG1MONTH__DIFFERENCE,SALES__YOY_GROWTH__QUARTERLY__VS__SALES__YOY_GROWTH__MONTHLY__LAG2MONTH__DIFFERENCE,SALES__YOY_GROWTH__QUARTERLY__VS__SALES__YOY_GROWTH__QUARTERLY__LAG2MONTH__DIFFERENCE,SALES__YOY_GROWTH__QUARTERLY__VS__COUNT__YOY_GROWTH__MONTHLY__LAG2MONTH__DIFFERENCE,SALES__YOY_GROWTH__QUARTERLY__VS__COUNT__YOY_GROWTH__QUARTERLY__LAG2MONTH__DIFFERENCE,SALES__YOY_GROWTH__QUARTERLY__VS__SALES__YOY_GROWTH__MONTHLY__LAG1QUARTER__DIFFERENCE,SALES__YOY_GROWTH__QUARTERLY__VS__SALES__YOY_GROWTH__QUARTERLY__LAG1QUARTER__DIFFERENCE,SALES__YOY_GROWTH__QUARTERLY__VS__COUNT__YOY_GROWTH__MONTHLY__LAG1QUARTER__DIFFERENCE,…,COUNT__YOY_GROWTH__QUARTERLY__VS__COUNT__YOY_GROWTH__QUARTERLY__LAG1QUARTER__DIFFERENCE,COUNT__YOY_GROWTH__QUARTERLY__VS__SALES__YOY_GROWTH__MONTHLY__LAG2QUARTER__DIFFERENCE,COUNT__YOY_GROWTH__QUARTERLY__VS__SALES__YOY_GROWTH__QUARTERLY__LAG2QUARTER__DIFFERENCE,COUNT__YOY_GROWTH__QUARTERLY__VS__COUNT__YOY_GROWTH__MONTHLY__LAG2QUARTER__DIFFERENCE,COUNT__YOY_GROWTH__QUARTERLY__VS__COUNT__YOY_GROWTH__QUARTERLY__LAG2QUARTER__DIFFERENCE,SALES__YOY_GROWTH__MONTHLY__VS__SALES__YOY_GROWTH__MONTHLY__LAG1MONTH__ZSCORE,SALES__YOY_GROWTH__MONTHLY__VS__SALES__YOY_GROWTH__QUARTERLY__LAG1MONTH__ZSCORE,SALES__YOY_GROWTH__MONTHLY__VS__SALES__YOY_GROWTH__MONTHLY__LAG2MONTH__ZSCORE,SALES__YOY_GROWTH__MONTHLY__VS__SALES__YOY_GROWTH__QUARTERLY__LAG2MONTH__ZSCORE,SALES__YOY_GROWTH__MONTHLY__VS__SALES__YOY_GROWTH__MONTHLY__LAG1QUARTER__ZSCORE,SALES__YOY_GROWTH__MONTHLY__VS__SALES__YOY_GROWTH__QUARTERLY__LAG1QUARTER__ZSCORE,SALES__YOY_GROWTH__MONTHLY__VS__SALES__YOY_GROWTH__MONTHLY__LAG2QUARTER__ZSCORE,SALES__YOY_GROWTH__MONTHLY__VS__SALES__YOY_GROWTH__QUARTERLY__LAG2QUARTER__ZSCORE,SALES__YOY_GROWTH__QUARTERLY__VS__SALES__YOY_GROWTH__MONTHLY__LAG1MONTH__ZSCORE,SALES__YOY_GROWTH__QUARTERLY__VS__SALES__YOY_GROWTH__QUARTERLY__LAG1MONTH__ZSC

In [18]:
feat_cols = [c for c in result.columns if "__VS__" in c]
result.select(feat_cols).drop_nulls().describe()

statistic,SALES__YOY_GROWTH__MONTHLY__VS__ZERO_BENCHMARK__DIFFERENCE,SALES__YOY_GROWTH__QUARTERLY__VS__ZERO_BENCHMARK__DIFFERENCE,COUNT__YOY_GROWTH__MONTHLY__VS__ZERO_BENCHMARK__DIFFERENCE,COUNT__YOY_GROWTH__QUARTERLY__VS__ZERO_BENCHMARK__DIFFERENCE,SALES__YOY_GROWTH__MONTHLY__VS__ZERO_BENCHMARK__ZSCORE,SALES__YOY_GROWTH__QUARTERLY__VS__ZERO_BENCHMARK__ZSCORE,COUNT__YOY_GROWTH__MONTHLY__VS__ZERO_BENCHMARK__ZSCORE,COUNT__YOY_GROWTH__QUARTERLY__VS__ZERO_BENCHMARK__ZSCORE,SALES__YOY_GROWTH__MONTHLY__VS__SALES__YOY_GROWTH__MONTHLY__LAG1MONTH__DIFFERENCE,SALES__YOY_GROWTH__MONTHLY__VS__SALES__YOY_GROWTH__QUARTERLY__LAG1MONTH__DIFFERENCE,SALES__YOY_GROWTH__MONTHLY__VS__COUNT__YOY_GROWTH__MONTHLY__LAG1MONTH__DIFFERENCE,SALES__YOY_GROWTH__MONTHLY__VS__COUNT__YOY_GROWTH__QUARTERLY__LAG1MONTH__DIFFERENCE,SALES__YOY_GROWTH__MONTHLY__VS__SALES__YOY_GROWTH__MONTHLY__LAG2MONTH__DIFFERENCE,SALES__YOY_GROWTH__MONTHLY__VS__SALES__YOY_GROWTH__QUARTERLY__LAG2MONTH__DIFFERENCE,SALES__YOY_GROWTH__MONTHLY__VS__COUNT__YOY_GROWTH__MONTHLY__LAG2MONTH__DIFFERENCE,SALES__YOY_GROWTH__MONTHLY__VS__COUNT__YOY_GROWTH__QUARTERLY__LAG2MONTH__DIFFERENCE,SALES__YOY_GROWTH__MONTHLY__VS__SALES__YOY_GROWTH__MONTHLY__LAG1QUARTER__DIFFERENCE,SALES__YOY_GROWTH__MONTHLY__VS__SALES__YOY_GROWTH__QUARTERLY__LAG1QUARTER__DIFFERENCE,SALES__YOY_GROWTH__MONTHLY__VS__COUNT__YOY_GROWTH__MONTHLY__LAG1QUARTER__DIFFERENCE,SALES__YOY_GROWTH__MONTHLY__VS__COUNT__YOY_GROWTH__QUARTERLY__LAG1QUARTER__DIFFERENCE,SALES__YOY_GROWTH__MONTHLY__VS__SALES__YOY_GROWTH__MONTHLY__LAG2QUARTER__DIFFERENCE,SALES__YOY_GROWTH__MONTHLY__VS__SALES__YOY_GROWTH__QUARTERLY__LAG2QUARTER__DIFFERENCE,SALES__YOY_GROWTH__MONTHLY__VS__COUNT__YOY_GROWTH__MONTHLY__LAG2QUARTER__DIFFERENCE,SALES__YOY_GROWTH__MONTHLY__VS__COUNT__YOY_GROWTH__QUARTERLY__LAG2QUARTER__DIFFERENCE,SALES__YOY_GROWTH__QUARTERLY__VS__SALES__YOY_GROWTH__MONTHLY__LAG1MONTH__DIFFERENCE,SALES__YOY_GROWTH__QUARTERLY__VS__SALES__YOY_GROWTH__QUARTERLY__LAG1MONTH__DIFFERENCE,SALES__YOY_GROWTH__QUARTERLY__VS__COUNT__YOY_GROWTH__MONTHLY__LAG1MONTH__DIFFERENCE,SALES__YOY_GROWTH__QUARTERLY__VS__COUNT__YOY_GROWTH__QUARTERLY__LAG1MONTH__DIFFERENCE,SALES__YOY_GROWTH__QUARTERLY__VS__SALES__YOY_GROWTH__MONTHLY__LAG2MONTH__DIFFERENCE,SALES__YOY_GROWTH__QUARTERLY__VS__SALES__YOY_GROWTH__QUARTERLY__LAG2MONTH__DIFFERENCE,SALES__YOY_GROWTH__QUARTERLY__VS__COUNT__YOY_GROWTH__MONTHLY__LAG2MONTH__DIFFERENCE,SALES__YOY_GROWTH__QUARTERLY__VS__COUNT__YOY_GROWTH__QUARTERLY__LAG2MONTH__DIFFERENCE,SALES__YOY_GROWTH__QUARTERLY__VS__SALES__YOY_GROWTH__MONTHLY__LAG1QUARTER__DIFFERENCE,SALES__YOY_GROWTH__QUARTERLY__VS__SALES__YOY_GROWTH__QUARTERLY__LAG1QUARTER__DIFFERENCE,SALES__YOY_GROWTH__QUARTERLY__VS__COUNT__YOY_GROWTH__MONTHLY__LAG1QUARTER__DIFFERENCE,SALES__YOY_GROWTH__QUARTERLY__VS__COUNT__YOY_GROWTH__QUARTERLY__LAG1QUARTER__DIFFERENCE,…,COUNT__YOY_GROWTH__QUARTERLY__VS__COUNT__YOY_GROWTH__QUARTERLY__LAG1QUARTER__DIFFERENCE,COUNT__YOY_GROWTH__QUARTERLY__VS__SALES__YOY_GROWTH__MONTHLY__LAG2QUARTER__DIFFERENCE,COUNT__YOY_GROWTH__QUARTERLY__VS__SALES__YOY_GROWTH__QUARTERLY__LAG2QUARTER__DIFFERENCE,COUNT__YOY_GROWTH__QUARTERLY__VS__COUNT__YOY_GROWTH__MONTHLY__LAG2QUARTER__DIFFERENCE,COUNT__YOY_GROWTH__QUARTERLY__VS__COUNT__YOY_GROWTH__QUARTERLY__LAG2QUARTER__DIFFERENCE,SALES__YOY_GROWTH__MONTHLY__VS__SALES__YOY_GROWTH__MONTHLY__LAG1MONTH__ZSCORE,SALES__YOY_GROWTH__MONTHLY__VS__SALES__YOY_GROWTH__QUARTERLY__LAG1MONTH__ZSCORE,SALES__YOY_GROWTH__MONTHLY__VS__SALES__YOY_GROWTH__MONTHLY__LAG2MONTH__ZSCORE,SALES__YOY_GROWTH__MONTHLY__VS__SALES__YOY_GROWTH__QUARTERLY__LAG2MONTH__ZSCORE,SALES__YOY_GROWTH__MONTHLY__VS__SALES__YOY_GROWTH__MONTHLY__LAG1QUARTER__ZSCORE,SALES__YOY_GROWTH__MONTHLY__VS__SALES__YOY_GROWTH__QUARTERLY__LAG1QUARTER__ZSCORE,SALES__YOY_GROWTH__MONTHLY__VS__SALES__YOY_GROWTH__MONTHLY__LAG2QUARTER__ZSCORE,SALES__YOY_GROWTH__MONTHLY__VS__SALES__YOY_GROWTH__QUARTERLY__LAG2QUARTER__ZSCORE,SALES__YOY_GROWTH__QUARTERLY__VS__SALES__YOY_GROWTH__MONTHLY__LAG1MONTH__ZSC

## 7. Preset Builders

In [19]:
from feature_kit.presets import sales_yoy_vs_zero, sales_rolling_vs_zero

params2 = Param(keys=["specid"], metrics=["SALES", "COUNT"])

yoy_result = sales_yoy_vs_zero(params2).compute(raw_df)
print("sales_yoy_vs_zero columns:", yoy_result.columns)

roll_result = sales_rolling_vs_zero(params2).compute(raw_df)
print("sales_rolling_vs_zero columns:", roll_result.columns)

sales_yoy_vs_zero columns: ['specid', 'DATADATE', 'SALES__YOY_GROWTH__QUADWEEKLY__VS__ZERO_BENCHMARK__DIFFERENCE', 'SALES__YOY_GROWTH__QUARTERLY__VS__ZERO_BENCHMARK__DIFFERENCE', 'COUNT__YOY_GROWTH__QUADWEEKLY__VS__ZERO_BENCHMARK__DIFFERENCE', 'COUNT__YOY_GROWTH__QUARTERLY__VS__ZERO_BENCHMARK__DIFFERENCE', 'SALES__YOY_GROWTH__QUADWEEKLY__VS__ZERO_BENCHMARK__SIGN', 'SALES__YOY_GROWTH__QUARTERLY__VS__ZERO_BENCHMARK__SIGN', 'COUNT__YOY_GROWTH__QUADWEEKLY__VS__ZERO_BENCHMARK__SIGN', 'COUNT__YOY_GROWTH__QUARTERLY__VS__ZERO_BENCHMARK__SIGN']
sales_rolling_vs_zero columns: ['specid', 'DATADATE', 'SALES__ROLLING_SUM__DAILY__VS__ZERO_BENCHMARK__DIFFERENCE', 'SALES__ROLLING_SUM__WEEKLY__VS__ZERO_BENCHMARK__DIFFERENCE', 'SALES__ROLLING_SUM__QUADWEEKLY__VS__ZERO_BENCHMARK__DIFFERENCE', 'SALES__ROLLING_SUM__QUARTERLY__VS__ZERO_BENCHMARK__DIFFERENCE', 'COUNT__ROLLING_SUM__DAILY__VS__ZERO_BENCHMARK__DIFFERENCE', 'COUNT__ROLLING_SUM__WEEKLY__VS__ZERO_BENCHMARK__DIFFERENCE', 'COUNT__ROLLING_SUM__QUADWE